In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive/credit-risk-assessment-system')

import pandas as pd
import numpy as np

X_train = pd.read_csv('data/processed/X_train_fe.csv')
X_test = pd.read_csv('data/processed/X_test_fe.csv')
y_train = pd.read_csv('data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('data/processed/y_test.csv').squeeze()

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(246008, 195) (61503, 195) (246008,) (61503,)


In [3]:
X_train_model = X_train.drop(columns=['SK_ID_CURR'])
X_test_model = X_test.drop(columns=['SK_ID_CURR'])

print(X_train_model.shape, X_test_model.shape)

(246008, 194) (61503, 194)


In [5]:
missing_check = X_train_model.isnull().sum()
missing_check = missing_check[missing_check > 0]
print(missing_check)

AMT_ANNUITY_RAW    10
dtype: int64


In [6]:
import numpy as np
inf_check = X_train_model.replace([np.inf, -np.inf], np.nan).isnull().sum()
inf_check = inf_check[inf_check > 0]
print(inf_check)

AMT_ANNUITY_RAW    10
dtype: int64


In [7]:
leftover_cols = [c for c in ['AMT_ANNUITY_RAW', 'AMT_CREDIT_RAW', 'AMT_INCOME_TOTAL_RAW'] if c in X_train_model.columns]
print("Dropping:", leftover_cols)

X_train_model = X_train_model.drop(columns=leftover_cols)
X_test_model = X_test_model.drop(columns=leftover_cols)

print(X_train_model.shape, X_test_model.shape)

Dropping: ['AMT_ANNUITY_RAW', 'AMT_CREDIT_RAW', 'AMT_INCOME_TOTAL_RAW']
(246008, 191) (61503, 191)


In [8]:
print("Train NaN:", X_train_model.isnull().sum().sum())
print("Test NaN:", X_test_model.isnull().sum().sum())
print("Train inf:", np.isinf(X_train_model.select_dtypes(include=[np.number])).sum().sum())
print("Test inf:", np.isinf(X_test_model.select_dtypes(include=[np.number])).sum().sum())

Train NaN: 0
Test NaN: 0
Train inf: 0
Test inf: 0


Step 2 — train a Logistic Regression baseline

In [10]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=3000, random_state=42)
log_reg.fit(X_train_model, y_train)

print("Training complete")

Training complete


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [11]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

y_pred = log_reg.predict(X_test_model)
y_pred_proba = log_reg.predict_proba(X_test_model)[:, 1]

auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC-ROC: {auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

AUC-ROC: 0.7410

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.56      0.01      0.02      4965

    accuracy                           0.92     61503
   macro avg       0.74      0.50      0.49     61503
weighted avg       0.89      0.92      0.88     61503

Confusion Matrix:
[[56498    40]
 [ 4915    50]]


In [12]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=8, random_state=42)
dt.fit(X_train_model, y_train)

y_pred_dt = dt.predict(X_test_model)
y_pred_proba_dt = dt.predict_proba(X_test_model)[:, 1]

print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba_dt):.4f}")
print(classification_report(y_test, y_pred_dt))

AUC-ROC: 0.7186
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.34      0.01      0.02      4965

    accuracy                           0.92     61503
   macro avg       0.63      0.50      0.49     61503
weighted avg       0.87      0.92      0.88     61503



In [14]:
os.makedirs('reports', exist_ok=True)

results.to_csv('reports/model_comparison_results.csv', index=False)
print("Saved successfully")

Saved successfully


In [15]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'AUC-ROC': [0.7410, 0.7186]
})
print(results)

results.to_csv('reports/model_comparison_results.csv', index=False)

                 Model  AUC-ROC
0  Logistic Regression   0.7410
1        Decision Tree   0.7186


In [16]:
print(os.getcwd())

/content/drive/MyDrive/credit-risk-assessment-system
